In [1]:
import torch
from torch import nn
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 
import os 
import plotly.express as px
from sklearn.model_selection import train_test_split
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from tqdm.auto import tqdm


In [2]:
data_dir = "/kaggle/input/multi-cancer/Multi Cancer/Multi Cancer"
target_folder = 'Brain Cancer'

filepath = []
labels = []
all_folder_path = os.path.join(data_dir, target_folder)

if os.path.isdir(all_folder_path):
    filelist = os.listdir(all_folder_path)
    for f in filelist:
        fpath = os.path.join(all_folder_path, f)
        fipath = os.listdir(fpath)
        for image in fipath:
            path = os.path.join(fpath, image)
            filepath.append(path)
            labels.append(f)
            
f_series = pd.Series(filepath, name='filepath')
l_series = pd.Series(labels, name='labels')
df = pd.concat([f_series, l_series], axis=1)
display(df)

,filepath,labels
0,/kaggle/input/multi-cancer/Multi Cancer/Multi ...,brain_tumor
1,/kaggle/input/multi-cancer/Multi Cancer/Multi ...,brain_tumor
2,/kaggle/input/multi-cancer/Multi Cancer/Multi ...,brain_tumor
3,/kaggle/input/multi-cancer/Multi Cancer/Multi ...,brain_tumor
4,/kaggle/input/multi-cancer/Multi Cancer/Multi ...,brain_tumor
...,...,...
14995,/kaggle/input/multi-cancer/Multi Cancer/Multi ...,brain_menin
14996,/kaggle/input/multi-cancer/Multi Cancer/Multi ...,brain_menin
14997,/kaggle/input/multi-cancer/Multi Cancer/Multi ...,brain_menin
14998,/kaggle/input/multi-cancer/Multi Cancer/Multi ...,brain_menin


In [3]:
count_df=df['labels'].value_counts().reset_index()
count_df.columns=['labels','count']
fig_df=px.bar(count_df,x='labels',y='count',title='count of labels in df',text_auto=True)
fig_df.show()

In [4]:
strat = df['labels']
train_df, dummy_df = train_test_split(df, test_size=0.2, stratify=strat)
strate = dummy_df['labels']
valid_df, test_df = train_test_split(dummy_df, test_size=0.5, stratify=strate)


In [5]:
count_train = train_df['labels'].value_counts().reset_index()
count_train.columns = ['labels', 'count']
fig_train = px.bar(count_train, x='labels', y='count', title='Count of labels in train_df', text_auto=True)
fig_train.show()

In [6]:
count_train = valid_df['labels'].value_counts().reset_index()
count_train.columns = ['labels', 'count']
fig_train = px.bar(count_train, x='labels', y='count', title='Count of labels in train_df', text_auto=True)
fig_train.show()

In [7]:
count_train = test_df['labels'].value_counts().reset_index()
count_train.columns = ['labels', 'count']
fig_train = px.bar(count_train, x='labels', y='count', title='Count of labels in train_df', text_auto=True)
fig_train.show()

In [8]:
# Path to your dataset
data_dir = "/kaggle/input/multi-cancer/Multi Cancer/Multi Cancer/Brain Cancer"

# Define transforms (resize + tensor + normalization)
transform = transforms.Compose([
    transforms.Resize((224, 224)),   # resize to fit pretrained models like ResNet
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], 
                         [0.229, 0.224, 0.225])  # standard ImageNet normalization
])


In [9]:

dataset = datasets.ImageFolder(root=data_dir, transform=transform)

print("Classes:", dataset.classes)   # should print ['brain_glioma', 'brain_menin', 'brain_tumour']
print("Number of images:", len(dataset))


Classes: ['brain_glioma', 'brain_menin', 'brain_tumor']
Number of images: 15000


In [10]:
from torch.utils.data import random_split

train_size = int(0.8*len(dataset))
val_size = (len(dataset) - train_size)//2
train_dataset, val_dataset, test_dataset = random_split(dataset, [train_size, val_size, val_size])

In [11]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=os.cpu_count())
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=os.cpu_count())
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=os.cpu_count())

## Base Model

In [12]:
class BrainCancerModelV0(nn.Module):
    def __init__(self, num_classes=3):
        super(BrainCancerModelV0, self).__init__()

        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, stride=1, padding=1)
        self.maxpool1 = nn.MaxPool2d(kernel_size=2)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1)
        self.maxpool2 = nn.MaxPool2d(kernel_size=2)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
        self.maxpool3 = nn.MaxPool2d(kernel_size=2)
        self.relu = nn.ReLU()
        self.fc1 = nn.Linear(28*28*64, 512)
        self.fc2 = nn.Linear(512, num_classes)
        self.dropout = nn.Dropout(0.5)

    def forward(self, x):
        x = self.maxpool1(self.relu(self.conv1(x)))
        x = self.maxpool2(self.relu(self.conv2(x)))
        x = self.maxpool3(self.relu(self.conv3(x)))
        x = x.reshape(x.shape[0], -1)
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x
        
        

In [13]:
device = torch.device("cuda" if torch.cuda.is_available() else 'cpu')
model = BrainCancerModelV0(num_classes=3).to(device)
print(model)

BrainCancerModelV0(
  (conv1): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (maxpool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (maxpool2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (maxpool3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (relu): ReLU()
  (fc1): Linear(in_features=50176, out_features=512, bias=True)
  (fc2): Linear(in_features=512, out_features=3, bias=True)
  (dropout): Dropout(p=0.5, inplace=False)
)


In [14]:
from torchinfo import summary

In [15]:
s = summary(model, input_size=(1, 3, 224, 224))
print(s)

Layer (type:depth-idx)                   Output Shape              Param #
BrainCancerModelV0                       [1, 3]                    --
├─Conv2d: 1-1                            [1, 16, 224, 224]         448
├─ReLU: 1-2                              [1, 16, 224, 224]         --
├─MaxPool2d: 1-3                         [1, 16, 112, 112]         --
├─Conv2d: 1-4                            [1, 32, 112, 112]         4,640
├─ReLU: 1-5                              [1, 32, 112, 112]         --
├─MaxPool2d: 1-6                         [1, 32, 56, 56]           --
├─Conv2d: 1-7                            [1, 64, 56, 56]           18,496
├─ReLU: 1-8                              [1, 64, 56, 56]           --
├─MaxPool2d: 1-9                         [1, 64, 28, 28]           --
├─Linear: 1-10                           [1, 512]                  25,690,624
├─ReLU: 1-11                             [1, 512]                  --
├─Dropout: 1-12                          [1, 512]                  --

In [16]:
x = torch.randn((3, 3, 224, 224)).to(device)
model(x)

tensor([[ 0.0306,  0.0317,  0.0114],
        [ 0.0274,  0.0371, -0.0259],
        [-0.0068, -0.0284, -0.0489]], device='cuda:0',
       grad_fn=<AddmmBackward0>)

In [17]:
def train_step(model, dataloader, loss_fn, optimizer, epochs=5, device='cpu'):
    model.train()
    train_acc, train_loss = 0, 0

    for X, y in dataloader:
        X, y = X.to(device), y.to(device)
        y_preds = model(X)
        loss = loss_fn(y_preds, y)
        train_loss += loss 
        y_pred_class = torch.argmax(torch.softmax(y_preds, dim=1), dim=1)
        train_acc += (y==y_pred_class).sum().item()/len(X)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        
    train_acc = train_acc/len(dataloader)
    train_loss = train_loss/len(dataloader)
    return train_loss, train_acc



def val_step(model, dataloader, loss_fn, device='cpu'):
    val_acc, val_loss = 0, 0
    with torch.inference_mode():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            y_preds = model(X)
            loss = loss_fn(y_preds, y)
            val_loss += loss 
            y_pred_class = torch.argmax(torch.softmax(y_preds, dim=1), dim=1)
            val_acc += (y==y_pred_class).sum().item()/len(X)
            
        val_acc = val_acc/len(dataloader)
        val_loss = val_loss/len(dataloader)
    return val_loss, val_acc

In [18]:
def train(model,
         train_dataloader,
         val_dataloader,
         optimizer,
         loss_fn,
         epochs=5,
         device=device):
    results = {
        "train_loss":[],
        "train_acc": [],
        "val_loss": [],
        "val_acc":[]
    }
    for epoch in tqdm(range(epochs)):
        train_loss, train_acc = train_step(
            model=model,
            dataloader=train_dataloader,
            loss_fn=loss_fn,
            optimizer=optimizer,
            device=device
        )
        val_loss, val_acc = val_step(
            model=model, 
            dataloader=val_dataloader,
            loss_fn=loss_fn,
            device=device
        )
        results["train_loss"].append(train_loss)
        results["train_acc"].append(train_acc)
        results["val_loss"].append(val_loss)
        results["val_acc"].append(val_acc)

        print(f"Epoch: {epoch} | Train Loss: {train_loss:.2f} | Train Accuracy: {train_acc:.2f} | Validation Loss: {val_loss:.2f} | Validation Accuarcy: {val_acc:.2f}")
    return results

In [19]:
device = "cuda" if torch.cuda.is_available() else "cpu"
NUM_EPOCHS = 10
model_0 = BrainCancerModelV0(num_classes=3)
# model_0 = nn.DataParallel(model_0)
model_0.to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params=model_0.parameters(), lr=0.01)



In [22]:
NUM_EPOCHS=50

## Base Model Training


In [23]:
from timeit import default_timer as timer
start_time = timer()
model_0_results = train(model=model_0,
                        train_dataloader=train_loader,
                        val_dataloader=val_loader,
                        optimizer=optimizer,
                        loss_fn=loss_fn,
                        epochs=NUM_EPOCHS,
                        device=device)
end_time = timer()
print(f"[INFO] Total training time: {end_time-start_time:.3f} seconds")

  0%|          | 0/50 [00:00<?, ?it/s]

Epoch: 0 | Train Loss: 0.27 | Train Accuracy: 0.90 | Validation Loss: 0.96 | Validation Accuarcy: 0.74
Epoch: 1 | Train Loss: 0.26 | Train Accuracy: 0.90 | Validation Loss: 0.86 | Validation Accuarcy: 0.77
Epoch: 2 | Train Loss: 0.23 | Train Accuracy: 0.92 | Validation Loss: 1.00 | Validation Accuarcy: 0.76
Epoch: 3 | Train Loss: 0.21 | Train Accuracy: 0.92 | Validation Loss: 1.01 | Validation Accuarcy: 0.73
Epoch: 4 | Train Loss: 0.21 | Train Accuracy: 0.93 | Validation Loss: 1.09 | Validation Accuarcy: 0.75
Epoch: 5 | Train Loss: 0.22 | Train Accuracy: 0.93 | Validation Loss: 1.18 | Validation Accuarcy: 0.75
Epoch: 6 | Train Loss: 0.24 | Train Accuracy: 0.92 | Validation Loss: 1.47 | Validation Accuarcy: 0.72
Epoch: 7 | Train Loss: 0.20 | Train Accuracy: 0.93 | Validation Loss: 1.45 | Validation Accuarcy: 0.76
Epoch: 8 | Train Loss: 0.19 | Train Accuracy: 0.93 | Validation Loss: 1.11 | Validation Accuarcy: 0.75
Epoch: 9 | Train Loss: 0.17 | Train Accuracy: 0.94 | Validation Loss: 1.0

## Main Model <br>
### This is a scaled down version of resnet with lesser parameters 
### Resnet50 ~ 25 Million parameters 
### BrainCancerModelV1 ~ 4 Million parameters 

In [86]:
class block(nn.Module):
    def __init__(self, in_channels, out_channels, identity_downsample=None, stride=1):
        super(block, self).__init__()
        self.expansion = 4
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=1, padding=0)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=stride, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.conv3 = nn.Conv2d(out_channels, out_channels*self.expansion, kernel_size=1, stride=1, padding=0)
        self.bn3 = nn.BatchNorm2d(out_channels*self.expansion)
        self.relu = nn.ReLU()
        self.identity_downsample = identity_downsample
        
    def forward(self, x):
        identity = x

        x = self.relu(self.bn1(self.conv1(x)))
        x = self.relu(self.bn2(self.conv2(x)))
        x = self.relu(self.bn3(self.conv3(x)))

        if self.identity_downsample is not None:
            identity = self.identity_downsample(identity)
        x = x +  identity
        x = self.relu(x)
        return x
        


class BrainCancerModelV1(nn.Module):

    def __init__(self, block, layers, image_channels, num_classes):
        super(BrainCancerModelV1, self).__init__()
        self.in_channels = 64
        self.conv1 = nn.Conv2d(image_channels, 64, kernel_size=7, stride=2, padding=3)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU()
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

        self.layer1 = self._make_layer(block, layers[0], out_channels=64, stride=1)
        self.layer2 = self._make_layer(block, layers[1], out_channels=128, stride=2)
        self.layer3 = self._make_layer(block, layers[2], out_channels=256, stride=2)

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(256*4, num_classes)

    def forward(self, x):
        x = self.maxpool(self.relu(self.bn1(self.conv1(x))))
        # x = self.avgpool(self.layer4(self.layer3(self.layer2(self.layer1(x)))))
        x = self.avgpool(self.layer3(self.layer2(self.layer1(x))))
        x = x.reshape(x.shape[0], -1)
        return self.fc(x)


    def _make_layer(self, block, num_residual_blocks, out_channels, stride):
        identity_downsample = None
        layers = []
        if stride != 1 or self.in_channels != out_channels*4:
            identity_downsample = nn.Sequential(
                nn.Conv2d(self.in_channels, out_channels*4, kernel_size=1, stride=stride),
                nn.BatchNorm2d(out_channels*4)
            )
            layers.append(block(self.in_channels, out_channels, identity_downsample, stride))
            self.in_channels = out_channels*4
            for i in range(num_residual_blocks-1):
                layers.append(block(self.in_channels, out_channels))
            return nn.Sequential(*layers)
            

    

In [87]:
def BrainNet50(img_channels=3, num_classes=3):
    return BrainCancerModelV1(block, [1, 2, 3], img_channels, num_classes)

In [88]:
def test():
    net = BrainNet50()
    x = torch.randn(1, 3, 224, 224)
    y = net(x)
    print(y.shape)
    print(summary(net, input_size=(1, 3, 224, 224)))
    
test()

torch.Size([1, 3])
Layer (type:depth-idx)                   Output Shape              Param #
BrainCancerModelV1                       [1, 3]                    --
├─Conv2d: 1-1                            [1, 64, 112, 112]         9,472
├─BatchNorm2d: 1-2                       [1, 64, 112, 112]         128
├─ReLU: 1-3                              [1, 64, 112, 112]         --
├─MaxPool2d: 1-4                         [1, 64, 56, 56]           --
├─Sequential: 1-5                        [1, 256, 56, 56]          --
│    └─block: 2-1                        [1, 256, 56, 56]          --
│    │    └─Conv2d: 3-1                  [1, 64, 56, 56]           4,160
│    │    └─BatchNorm2d: 3-2             [1, 64, 56, 56]           128
│    │    └─ReLU: 3-3                    [1, 64, 56, 56]           --
│    │    └─Conv2d: 3-4                  [1, 64, 56, 56]           36,928
│    │    └─BatchNorm2d: 3-5             [1, 64, 56, 56]           128
│    │    └─ReLU: 3-6                    [1, 64, 56, 

In [91]:
device = "cuda" if torch.cuda.is_available() else "cpu"
NUM_EPOCHS = 30
model_0 = BrainNet50()
model_0.to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params=model_0.parameters(), lr=0.01)



In [ ]:
NUM_EPOCHS = 30

## Main Model Training 

In [103]:
from timeit import default_timer as timer
start_time = timer()
model_0_results = train(model=model_0,
                        train_dataloader=train_loader,
                        val_dataloader=val_loader,
                        optimizer=optimizer,
                        loss_fn=loss_fn,
                        epochs=NUM_EPOCHS,
                        device=device)
end_time = timer()
print(f"[INFO] Total training time: {end_time-start_time:.3f} seconds")

  0%|          | 0/30 [00:00<?, ?it/s]

Epoch: 0 | Train Loss: 0.02 | Train Accuracy: 0.99 | Validation Loss: 0.08 | Validation Accuarcy: 0.98
Epoch: 1 | Train Loss: 0.02 | Train Accuracy: 0.99 | Validation Loss: 0.07 | Validation Accuarcy: 0.98
Epoch: 2 | Train Loss: 0.01 | Train Accuracy: 1.00 | Validation Loss: 0.02 | Validation Accuarcy: 0.99
Epoch: 3 | Train Loss: 0.02 | Train Accuracy: 0.99 | Validation Loss: 0.08 | Validation Accuarcy: 0.98
Epoch: 4 | Train Loss: 0.02 | Train Accuracy: 0.99 | Validation Loss: 0.07 | Validation Accuarcy: 0.98
Epoch: 5 | Train Loss: 0.01 | Train Accuracy: 1.00 | Validation Loss: 0.05 | Validation Accuarcy: 0.98
Epoch: 6 | Train Loss: 0.01 | Train Accuracy: 1.00 | Validation Loss: 0.04 | Validation Accuarcy: 0.99
Epoch: 7 | Train Loss: 0.00 | Train Accuracy: 1.00 | Validation Loss: 0.03 | Validation Accuarcy: 0.99
Epoch: 8 | Train Loss: 0.03 | Train Accuracy: 0.99 | Validation Loss: 0.04 | Validation Accuarcy: 0.98
Epoch: 9 | Train Loss: 0.00 | Train Accuracy: 1.00 | Validation Loss: 0.0

In [94]:
from pathlib import Path

def save_model(model:torch.nn.Module,
               target_dir: str,
               model_name: str):
    target_dir_path = Path(target_dir)
    target_dir_path.mkdir(parents=True,
                          exist_ok=True)

    assert model_name.endswith("pth") or model_name.endswith(".pt"), "model should end with pth or pt"
    model_save_path = target_dir_path / model_name
    print(f"[INFO] Saving model to {model_save_path}")
    torch.save(obj=model.state_dict(),
               f=model_save_path)

In [104]:
save_model(model=model_0,
           target_dir="models",
           model_name="BrainCancerModel1_ver2.pth")

[INFO] Saving model to models/BrainCancerModel1_ver2.pth


In [105]:
test_loss, test_acc = val_step(model_0, test_loader, loss_fn, device)

In [106]:
print(f"Loss in Testing data: {test_loss:.2f}")
print(f"Testing Accuracy: {test_acc*100:.2f}%")

Loss in Testing data: 0.08
Testing Accuracy: 98.07%
